In [ ]:
from vizdoom import *
import cv2
import numpy as np
import time
import random
from matplotlib import pyplot as plt
import pandas as pd

# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

In [ ]:
# Ollama imports
import requests
import json

In [ ]:
# Instantiate TakeCover environment
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'

In [ ]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)

        # Reward shaping parameters
        # Positve reward for staying alive, negative reward for getting hit
        self.game.set_living_reward(1.0)
        self.game.set_death_penalty(100.0)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            obs    = self._process_frame(self.game.get_state().screen_buffer)
            health = self.game.get_state().game_variables[0]
            info   = {"health": health}
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return obs, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)

        
        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

In [ ]:
# env = VizDoomGym(render=False)
# obs, _ = env.reset()

In [ ]:
# delta_y_values = []
# delta_x_values = []
# for _ in range(500):
#     action = env.action_space.sample()
#     obs, reward, terminated, truncated, info = env.step(action)
    
#     # stampa i delta_x grezzi per calibrare le soglie
#     state = env.game.get_state()
#     if state:
#         for label in state.labels:
#             if label.object_name == "DoomImpBall":
#                 delta_x_values.append(-label.object_position_x)
#                 delta_y_values.append(label.object_position_y - state.game_variables[2])

#     if terminated or truncated:
#         obs, _ = env.reset()
        
# # Evaluate results to determine good thresholds for urgency classification
# max_delta_x = max(delta_x_values) 
# threshold_imminent   = max_delta_x * 0.5 
# threshold_approaching = max_delta_x

# print(f"Suggested URGENCY_IMMINENT threshold: {threshold_imminent:.2f}")
# print(f"Suggested URGENCY_APPROACHING threshold: {threshold_approaching:.2f}")

In [ ]:
class PerceptionLayer:
    URGENCY_IMMINENT   =  200
    URGENCY_APPROACHING = 400
    SIDE_THRESHOLD = 30 

    ACTION_NAMES = {0: "move_left", 1: "move_right"}

    def __init__(self):
        self.prev_health  = 100
        self.last_action  = None
        self.step_count   = 0

    def reset(self):
        self.prev_health = 100
        self.last_action = None
        self.step_count  = 0

    def update(self, state):
        self.step_count += 1

        player_y, health, delta_h = self._get_player_state(state)
        projectiles = self._get_projectiles(state, player_y)

        return self._serialize(player_y, health, delta_h, projectiles)
    
    def update_action(self, action):
        self.last_action = action
    
    def _get_player_state(self, state):
        vars = state.game_variables
        health = vars[0]
        player_y = vars[2]

        delta_h = health - self.prev_health
        self.prev_health = health

        return player_y, health, delta_h
    
    def _get_projectiles(self, state, player_y):
        projectiles = []
        for label in state.labels:
            if label.object_name == "DoomImpBall":
                delta_y = label.object_position_y - player_y
                delta_x = label.object_position_x
                projectiles.append({
                "delta_y": delta_y,
                "delta_x": delta_x,
                "side":    self._side(delta_y),
                "urgency": self._urgency(delta_x),
                })
        
        # Returns projectiles sorted by urgency (closest to player first)
        return sorted(projectiles, key=lambda p: abs(p["delta_x"]))
    
    def _side(self, delta_y: float):
        if abs(delta_y) < self.SIDE_THRESHOLD:
            return "center"
        return "left" if delta_y < 0 else "right"

    def _urgency(self, delta_x: float):
        dist = abs(delta_x)
        if dist < self.URGENCY_IMMINENT:
            return "imminent"
        elif dist < self.URGENCY_APPROACHING:
            return "approaching"
        return "distant"
    
    def _serialize(self, player_y, health, delta_h, projectiles):
        lines = []
        lines.append(f"Health: {int(health)}/100" + (f" (lost {abs(int(delta_h))} HP this step)" if delta_h < 0 else ""))
        lines.append(f"Player position: {self._side(player_y)}")
        lines.append(f"Last action: {self.ACTION_NAMES.get(self.last_action, 'none')}")
        lines.append(f"Survived: {self.step_count} steps")

        if not projectiles:
            lines.append("No projectiles visible.")
        else:
            lines.append(f"{len(projectiles)} projectile(s) detected:")
            for i, p in enumerate(projectiles, 1):
                lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side")

        return "\n".join(lines)

In [ ]:
# env = VizDoomGym(render=True)
# perception = PerceptionLayer()
# episodes = 1

# for episode in range(episodes):
#     obs, info = env.reset()
#     perception.reset()
#     truncated = False
#     terminated = False

#     while not terminated:
#         state = env.game.get_state()
#         state_text = perception.update(state, action)
#         action = random.choice([0, 1])  # Randomly choose to move left or right

#         print(state_text)

#         obs, reward, terminated, truncated, info = env.step(action)
#         perception.update(state, action)
#         time.sleep(0.1)
#     time.sleep(2)

# env.close()

In [ ]:
class LLMAgent:
    def __init__(self, model_name, backend_url):
        self.backend_url = backend_url
        self.model_name = model_name
        self.parse_failures = 0

        self.payload = {
            "model" : self.model_name,
            "prompt" : "",
            "stream" : False,
        }

    def decide(self, state_text):
        prompt  = self._build_prompt(state_text)
        response = self._call_llm(prompt)
        action  = self._parse_action(response)

        return action
    
    def _build_prompt(self, state_text):
        return f"""You are the tactical AI controller for an agent in a combat video game. Your primary objective is to ensure the agent's survival by effectively dodges the fireballs
            incoming from the enemies. Based on the provided GAME STATE description, you must decide the single best next tactical action for the agent.
            
            ### ALLOWED ACTIONS:
            You can only choose ONE of the following actions:
            - MOVE_LEFT: The player moves left for 4 game tics.
            - MOVE_RIGHT: The player moves right for 4 game tics.
            
            ### GAME STATE:
            {state_text}
            
            ### DECISION CRITERIA:
            - If a projectile is IMMINENT on your RIGHT side, move LEFT immediately.
            - If a projectile is IMMINENT on your LEFT side, move RIGHT immediately.
            - If multiple projectiles are detected, prioritize the closest (listed first).
            - If no projectiles are visible, move toward the CENTER of the arena.
            - If you just lost HP, change direction — your current position is dangerous.
            
            ### OUTPUT FORMAT:
            Respond with EXACTLY one of these two strings, nothings else: MOVE_LEFT or MOVE_RIGHT"""

    def _call_llm(self, prompt):
        self.payload["prompt"] = prompt
        self.payload["stream"] = False

        try:
            response = requests.post(self.backend_url, json= self.payload)

            # Check response status
            if response.status_code == 200:
                return response.json()["response"]
            else:
                print(f"Ollama returned error: {response.status_code}")
                return ""
        except requests.exceptions.ConnectionError:
            print("Failed to connect to Ollama backend.")
            return ""   
    
    def _parse_action(self, response):
        clean = response.strip().upper()
        if "MOVE_LEFT" in clean:
            return 0
        elif "MOVE_RIGHT" in clean:
            return 1
        else:
            # fallback: random action if parsing fails + log the failure
            self.parse_failures += 1
            print(f"LLM response parsing failed. Total failures: {self.parse_failures}")
            return random.choice([0, 1])

#### Testing the LLM Policy Loop

In [ ]:
env = VizDoomGym(render=True)
perception = PerceptionLayer()
agent = LLMAgent(model_name="llama3.2:3b", backend_url="http://localhost:11434/api/generate")
episodes = 100

for episode in range(episodes):
    obs, info = env.reset()
    perception.reset()
    truncated = False
    terminated = False

    while not terminated and not truncated:
        # Get current game state and convert to text description, then pass to LLM for decision
        state = env.game.get_state()
        state_text = perception.update(state)
        action = agent.decide(state_text)

        # Execute the chosen action in the environment and observe the results
        obs, reward, terminated, truncated, info = env.step(action)

        perception.update_action(action)
        # time.sleep(0.1)
    time.sleep(2)

env.close()

In [ ]:
env.close()